# Step 4: Training Loop & Gradient Descent

Steps 2-3 built the pieces: a forward pass that makes a prediction, and backpropagation that computes how each weight should change to make that prediction less wrong. This step puts them to work, repeatedly, until the network actually learns to recognize digits.

**What is gradient descent, in plain terms?** The gradient (from backpropagation) points in the direction that would make the loss *worse*. So to make things *better*, take a small step in the *opposite* direction, for every weight, at the same time. Do that over and over, and the loss gradually goes down - like walking downhill by always stepping in the steepest downward direction you can currently see, without knowing what the whole landscape looks like.

In [1]:
from network import init_params, forward
from train import train, accuracy
from mnist_loader import load_mnist

X_train, y_train, X_test, y_test = load_mnist()
W1, b1, W2, b2 = init_params()

## Three settings that control training ("hyperparameters")

These aren't learned by the network - they're choices made ahead of time that control *how* learning happens:

- **Learning rate** (`0.5` here): how big a step to take in the gradient's direction each update. Too small and training crawls; too large and it can overshoot and never settle down (imagine taking huge downhill steps and bouncing past the bottom of the valley).
- **Batch size** (`64`): how many images to look at before taking one update step. Looking at all 60,000 before every single update would be slow and update the weights only once per epoch; looking at just 1 at a time is fast per step but each step is a noisy, unreliable estimate. 64 is a common middle ground.
- **Epochs** (`15`): how many full passes through the entire training set to make. Each pass gives every image another chance to nudge the weights.

## The training loop

For each epoch: shuffle the training data, split it into batches of 64, and for every batch run forward pass -> loss -> backward pass -> update every weight a small step downhill. `train()` (in [`train.py`](train.py)) does exactly this and records the loss and accuracy after every epoch so progress can be tracked.

In [2]:
W1, b1, W2, b2, history = train(
    X_train, y_train, W1, b1, W2, b2,
    epochs=15, batch_size=64, learning_rate=0.5,
)

Epoch 1/15 - loss: 0.2423 - train accuracy: 97.0%


Epoch 2/15 - loss: 0.1057 - train accuracy: 97.2%


Epoch 3/15 - loss: 0.0737 - train accuracy: 97.8%


Epoch 4/15 - loss: 0.0569 - train accuracy: 98.4%


Epoch 5/15 - loss: 0.0452 - train accuracy: 98.9%


...

Epoch 11/15 - loss: 0.0102 - train accuracy: 99.8%


Epoch 12/15 - loss: 0.0079 - train accuracy: 99.8%


Epoch 13/15 - loss: 0.0054 - train accuracy: 100.0%


Epoch 14/15 - loss: 0.0042 - train accuracy: 100.0%


Epoch 15/15 - loss: 0.0030 - train accuracy: 100.0%


The loss drops sharply in the first few epochs and keeps shrinking, while training accuracy climbs from ~97% to 100% by the end. That's exactly the shape a working training loop should produce - if the loss went up, or bounced around wildly, or never moved, that would point to a bug (or a learning rate set too high/low).

## Checking on genuinely unseen data

Training accuracy alone can be misleading - a network can partly *memorize* training examples rather than *learn general patterns*. The test set (10,000 images the network never trained on) is the real measure of whether it actually learned to recognize digits, not just these specific 60,000 images.

In [3]:
test_probs, _ = forward(X_test, W1, b1, W2, b2)
test_accuracy = accuracy(test_probs, y_test)

print(f"Final training accuracy: {history['accuracy'][-1]*100:.1f}%")
print(f"Test accuracy (unseen data): {test_accuracy*100:.1f}%")

Final training accuracy: 100.0%
Test accuracy (unseen data): 98.1%


Training accuracy reached 100%, test accuracy is a bit lower - a small, expected gap. This is mild **overfitting**: with enough epochs, the network started fitting quirks specific to the training images, not just the general shape of each digit. A 100%-vs-test gap this small isn't alarming, but it's worth naming honestly rather than only reporting the higher number. Step 5 looks at this in more detail with loss curves and a proper evaluation.

## Saving the trained network

So Step 5 can evaluate this exact trained network without re-running training from scratch.

In [4]:
import numpy as np
from pathlib import Path

Path("models").mkdir(exist_ok=True)
np.savez(
    "models/trained_network.npz",
    W1=W1, b1=b1, W2=W2, b2=b2,
    loss_history=history["loss"], accuracy_history=history["accuracy"],
)
print("Saved to models/trained_network.npz")

Saved to models/trained_network.npz


## Summary

- **Gradient descent**: take a small step in the opposite direction of the gradient, for every weight, to reduce the loss.
- **Mini-batch training**: update weights after every small batch of 64 images rather than the whole dataset, for far more frequent (if noisier) updates.
- After 15 epochs: **100% training accuracy, 98.1% test accuracy** - a working, genuinely trained network, with a small honest gap between the two worth discussing rather than hiding.

Next: Step 5 evaluates this network properly - loss curves, per-digit accuracy, and a sanity-check comparison against scikit-learn's own MLP implementation.